# Train one segment price model

Set `SEGMENT` in the next cell and run the notebook top to bottom. Run it once per segment:
`apartment_sale`, `apartment_rent`, `house_sale`, `house_rent`.

Everything behind this notebook lives in `src/inmoai_lt/modeling/`; the notebook only chooses a
segment, looks at the data, and reads the results.

Prerequisite: `python -m inmoai_lt clean` has been run, so `data/processed/segments/` exists.

In [1]:
SEGMENT = "apartment_sale"  # apartment_sale | apartment_rent | house_sale | house_rent

# None -> use the target in config/modeling.yaml.
# Override to compare modes: "price" | "price_per_sqm" | "log_price".
# predict() always returns EUR, so metrics stay comparable across all three.
TARGET = None

In [2]:
import pandas as pd
import plotly.express as px

from inmoai_lt.modeling import SegmentModel, format_metrics, load_model_config, load_segment

config = load_model_config(target=TARGET)
df = load_segment(SEGMENT)

print(f"{SEGMENT}: {len(df):,} listings | target = {config.target}")

apartment_sale: 3,036 listings | target = price


In [3]:
df

,listing_id,property_type,listing_type,segment,source_file,district,street,house_number,full_address,latitude,...,effective_year,years_since_renovation,is_new_build,floor_ratio,is_ground_floor,is_top_floor,area_per_room,plot_to_building_ratio,has_water_body_nearby,days_since_update
0,1-2796291,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Pašilaičiai,Girulių g.,12,"Vilnius, Pašilaičiai, Girulių g.",54.7380,...,2007,NaN,False,1.0000,False,True,32.0000,NaN,False,50
1,1-2999625,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Pilaitė,Vydūno g.,19,"Vilnius, Pilaitė, Vydūno g.",54.7070,...,1994,NaN,False,0.6667,False,False,23.5133,NaN,False,5
2,1-3010389,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Pilaitė,M. Jankaus g.,NaN,"Vilnius, Pilaitė, M. Jankaus g.",54.7125,...,2007,NaN,False,0.8571,False,False,25.0000,NaN,False,18
3,1-3057747,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Pilaitė,M. Mažvydo g.,1,"Vilnius, Pilaitė, M. Mažvydo g.",54.7120,...,2018,NaN,False,0.4000,False,False,25.5000,NaN,False,9
4,1-3094223,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Markučiai,Pakraščio g.,4,"Vilnius, Markučiai, Pakraščio g.",54.6755,...,1940,NaN,False,1.0000,False,True,20.0000,NaN,False,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3031,1-3693693,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Šnipiškės,Rinktinės g.,42,"Vilnius, Šnipiškės, Rinktinės g.",54.6990,...,1968,NaN,False,1.0000,False,True,17.0000,NaN,False,3
3032,1-3693701,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Antakalnis,Nemenčinės pl.,4D,"Vilnius, Antakalnis, Nemenčinės pl.",54.7248,...,2017,NaN,False,0.6667,False,False,27.5050,NaN,False,3
3033,1-3693703,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Viršuliškės,Viršuliškių g.,35,"Vilnius, Viršuliškės, Viršuliškių g.",54.7098,...,1977,NaN,False,0.8889,False,False,32.7300,NaN,False,3
3034,1-3693711,apartment,sale,apartment_sale,apartments_sale_vilnius.csv,Užupis,Užupio g.,19,"Vilnius, Užupis, Užupio g.",54.6808,...,1940,NaN,False,0.5000,False,False,29.6000,NaN,False,3


## 1. Inspect the segment

`price_eur` is the asking price for sale segments and EUR/month for rent segments.

In [4]:
summary = pd.DataFrame(
    {
        "price_eur": df["price_eur"].describe(),
        "total_area_sqm": df["total_area_sqm"].describe(),
        "price_per_sqm_eur": (df["price_eur"] / df["total_area_sqm"]).describe(),
    }
)
summary.round(1)

,price_eur,total_area_sqm,price_per_sqm_eur
count,3036.0,3036.0,3036.0
mean,269483.4,61.8,4189.3
std,272504.1,45.4,1583.3
min,12466.0,15.0,517.7
25%,145000.0,40.0,3078.5
50%,199500.0,54.0,3908.5
75%,298665.5,70.3,4984.0
max,7000000.0,1319.0,11900.0


In [5]:
df["district"].value_counts().head(15).to_frame("listings")

,listings
district,
Naujamiestis,377
Senamiestis,359
Žirmūnai,198
Šnipiškės,187
Pašilaičiai,182
Antakalnis,150
Pilaitė,146
Lazdynai,102
Karoliniškės,101


## 2. Train

`SegmentModel.train` picks the feature list from the property type, fits on a train split and
scores the held-out split. The saved model is the one that was scored -- there is no refit.

In [6]:
model = SegmentModel.train(SEGMENT, df=df, config=config)

print(f"train rows: {model.metadata['n_train']:,}   held-out rows: {model.metadata['n_test']:,}")
print(f"features:   {len(model.feature_spec.all_columns)}")
if model.low_n:
    print(
        "\nLOW-N: too few training rows for stable estimates. Metrics below will swing between"
        "\nseeds, and cap rates derived from this model are flagged low_confidence."
    )

train rows: 2,428   held-out rows: 608
features:   33


## 3. Evaluate on the held-out split

All amounts are EUR (EUR/month for rent), whichever target mode was used.

In [7]:
format_metrics(model.metrics)

,n,MAE (EUR),MedAE (EUR),RMSE (EUR),MAPE (%),RMSPE (%),R2
0,608,37555.7,17811.6,75787.4,16.27,35.67,0.8868


In [7]:
holdout = df.iloc[model.metadata["test_index"]]
actual = holdout["price_eur"].to_numpy(dtype=float)
predicted = model.predict(holdout)

fig = px.scatter(
    x=actual,
    y=predicted,
    hover_name=holdout["district"],
    opacity=0.5,
    labels={"x": "actual (EUR)", "y": "predicted (EUR)"},
    title=f"{SEGMENT}: predicted vs actual, held-out split",
)
limit = float(max(actual.max(), predicted.max()))
fig.add_shape(type="line", x0=0, y0=0, x1=limit, y1=limit, line={"dash": "dash"})
fig.show()

In [8]:
# Where the model is weakest. Districts with few held-out rows are noise, so they are dropped.
errors = pd.DataFrame(
    {
        "district": holdout["district"].to_numpy(),
        "abs_pct_error": abs(predicted - actual) / actual * 100,
    }
)
by_district = errors.groupby("district").agg(
    listings=("abs_pct_error", "size"),
    median_abs_pct_error=("abs_pct_error", "median"),
)
by_district[by_district["listings"] >= 10].sort_values("median_abs_pct_error").round(1)

,listings,median_abs_pct_error
district,,
Fabijoniškės,19,4.0
Antakalnis,40,5.2
Justiniškės,16,5.3
Pašilaičiai,29,5.8
Pilaitė,36,6.7
Šeškinė,20,7.0
Žirmūnai,43,7.8
Karoliniškės,18,7.8
Šnipiškės,41,8.0


## 4. Save

Writes `models/<segment>.joblib`. Reload anywhere with `SegmentModel.load(SEGMENT)`.
`02_cap_rate.ipynb` expects the sale and rent models of a property type to both be saved.

In [9]:
path = model.save()
print(f"saved -> {path}")

saved -> C:\Users\wn686\OneDrive - Deutsche Börse AG\Desktop\REPOs\InmoAI-lt\models\apartment_sale.joblib
